# Quickstart

This notebook shows the main ways to run modde:

1. minimise a Python function with `fmin`;
2. run on benchmark problems from [IOHexperimenter](https://iohprofiler.github.io/) (ioh);
3. log a run, including modde's internal state, for IOHanalyzer;
4. step through a run yourself.

In [ ]:
import numpy as np
import ioh
import modde

print("modde", modde.__version__)

## 1. Minimise a function

`fmin` needs the function, a budget (number of evaluations), the dimension and
the bounds. Without further settings it runs **classic DE** (DE/rand/1/bin).

In [ ]:
def sphere(x):
    return float(np.sum(x ** 2))

result = modde.fmin(sphere, budget=20_000, dim=10, lb=-5, ub=5, seed=42)
print("best value:", result.f)
print("evaluations:", result.evaluations)

The result also unpacks as a tuple: `x, f, evaluations, optimizer = modde.fmin(...)`.

A well-known variant is one argument away (see the next notebook for all presets):

In [ ]:
result = modde.fmin(sphere, budget=20_000, dim=10, lb=-5, ub=5, seed=42, preset="L-SHADE")
print("L-SHADE best value:", result.f)

## 2. Your function is wrapped as an ioh problem

modde wrapped `sphere` as an ioh problem behind the scenes:

In [ ]:
problem = result.optimizer.problem
print(type(problem).__name__, "| evaluations counted by the problem:", problem.state.evaluations)

Because **the problem counts the evaluations**, the budget can never be exceeded
by accident. The bigger benefit is that everything ioh offers now works,
including logging. ioh's benchmark suites (BBOB, CEC2013, CEC2022, ...) can be
passed directly; bounds and the stopping target come from the problem.

In [ ]:
f15 = ioh.get_problem(15, instance=1, dimension=5)   # BBOB Rastrigin
opt = modde.ModularDE(f15, budget=20_000, preset="L-SHADE", seed=1).run()
print("precision reached:", opt.best_f - f15.optimum.y, "after", f15.state.evaluations, "evaluations")

## 3. Logging a run, including internal state

To log, wrap your function yourself with `modde.wrap_problem` (or use an ioh
problem), attach an ioh logger and tell it which attributes of the optimizer to
**watch**. With `track_evaluations=True` the values are recorded for every evaluation.
The output folder can be loaded in [IOHanalyzer](https://iohanalyzer.liacs.nl).

In [ ]:
import tempfile, pathlib

problem = modde.wrap_problem(sphere, dim=5, lb=-5, ub=5, name="sphere")
folder = tempfile.mkdtemp()
logger = ioh.logger.Analyzer(root=folder, folder_name="modde-run", algorithm_name="L-SHADE")

opt = modde.ModularDE(problem, budget=2000, preset="L-SHADE", seed=3, track_evaluations=True)
logger.watch(opt, ["curr_F", "curr_CR", "corrected", "popstd"])
problem.attach_logger(logger)
opt.run()
problem.detach_logger()
logger.close()

data_file = next(pathlib.Path(folder).rglob("*.dat"))
print("\n".join(data_file.read_text().splitlines()[:6]))

## 4. Stepping through a run

`step()` runs one generation and returns `False` when the run is finished. In
between, the whole state can be inspected: `population` (an `(n, d)` array),
`fitness`, the `archive`, the SHADE `memory_F`, the population size `lambda_`, and so on.

In [ ]:
opt = modde.ModularDE(sphere, budget=6000, dim=5, lb=-5, ub=5, preset="L-SHADE", seed=4)
history = []
while opt.step():
    history.append((opt.evaluations, opt.lambda_, opt.best_f, opt.F_memory_mean))

for evals, lam, best, mean_F in history[::40]:
    print(f"{evals:6d} evaluations | population {lam:3d} | best {best:9.2e} | mean F memory {mean_F:.3f}")

Each generation is made of steps that can also be called one by one
(`sample_parameters`, `mutate`, `crossover`, `correct_bounds`, `evaluate`,
`select`, `adapt`). See [Concepts](../concepts.md).